# Lab 2: Q-Learning Agent for Cliff Walking

**Student ID:** 041107730

---

## Demo Instructions for Teacher

This notebook demonstrates Q-Learning algorithm solving the Cliff Walking problem.

**How to run:**
1. Click `Kernel` -> `Restart & Run All` to run the entire notebook
2. Or run each cell sequentially from top to bottom using `Shift + Enter`

**Sections:**
- Section 1-3: Setup (hyperparameters, imports, environment)
- Section 4-6: Core algorithm (Q-table, action selection, Q-value update)
- Section 7: Training (watch the agent learn!)
- Section 8-10: Results (testing, analysis, visualization)

**Expected Result:** After training, the agent learns to navigate safely above the cliff to reach the goal.

---

## Core Formula

$$Q(s,a) = r + \gamma \cdot \max_{a'} Q(s',a')$$

| Symbol | Name | Description |
|--------|------|-------------|
| $Q(s,a)$ | Q-value | Expected cumulative reward for taking action $a$ in state $s$ |
| $s$ | State | Current position of the agent |
| $a$ | Action | Action taken (LEFT, RIGHT, UP, DOWN) |
| $r$ | Reward | Immediate reward received after taking action |
| $\gamma$ | Gamma | Discount factor, controls importance of future rewards (0-1) |
| $s'$ | Next State | State after taking action $a$ |
| $\max_{a'} Q(s',a')$ | Max Future Q | Maximum Q-value achievable from next state |

## 1. Hyperparameters

Each hyperparameter controls a specific behavior. See comparisons below to understand their effects.

In [ ]:
# ============================================================
# EPISODES - 训练回合数 / Number of training episodes
# ============================================================
# 对比 Comparison:
#   EPISODES = 10  → 训练不足，可能还没学会
#   EPISODES = 50  → 通常足够学会基本策略 ✓
#   EPISODES = 500 → 更充分，但耗时更长
EPISODES = 50


# ============================================================
# GAMMA (γ) - 折扣因子 / Discount Factor
# ============================================================
# 作用：决定智能体多看重"未来奖励" vs "眼前奖励"
# 公式：Q(s,a) = r + γ × max Q(s',a')
#                    ↑
#               γ 乘以未来的 Q 值
#
# 对比 Comparison (假设未来能拿 +10 奖励):
#   γ = 0.0  → 0.0 × 10 = 0  → 只看眼前，完全忽略未来
#              像只看今天的人，从不存钱
#   γ = 0.5  → 0.5 × 10 = 5  → 未来奖励打五折
#              像短视的人
#   γ = 0.9  → 0.9 × 10 = 9  → 未来几乎和眼前一样重要 ✓
#              像有远见的人
#   γ = 1.0  → 1.0 × 10 = 10 → 未来 = 现在（可能不稳定）
GAMMA = 0.9


# ============================================================
# EPSILON (ε) - 探索率 / Exploration Rate
# ============================================================
# 作用：多大概率"随机尝试" vs "用已知最优"
#
# 对比 Comparison:
#   ε = 0.0  → 0% 随机 → 永不探索，可能错过更好的路
#              像固执的人，只走老路
#   ε = 0.1  → 10% 随机，90% 最优 → 偶尔探索 ✓
#              像务实的人，偶尔尝新
#   ε = 0.5  → 50% 随机 → 太随机，学得慢
#   ε = 1.0  → 100% 随机 → 纯瞎走，学不会
#              像无头苍蝇
EPSILON = 0.1


# ============================================================
# DECAY - 探索率衰减 / Exploration Decay
# ============================================================
# 作用：每回合后 ε 减少多少
# 公式：new_ε = old_ε - DECAY × old_ε
#
# 对比 Comparison (初始 ε=0.1):
#   DECAY = 0.0  → ε 保持 0.1 不变
#   DECAY = 0.5  → ε: 0.1 → 0.05 → 0.025 → ... ✓
#                  前期多探索，后期多利用
#   DECAY = 1.0  → ε 立刻变 0
DECAY = 0.5


# ============================================================
# ALPHA (α) - 学习率 / Learning Rate
# ============================================================
# 作用：新知识替换多少旧知识
# 完整公式：Q = (1-α)×Q_old + α×(r + γ×maxQ')
#              ↑保留旧知识    ↑加入新知识
#
# 对比 Comparison:
#   α = 0.0  → 100% 保留旧 → 永远不学习
#   α = 0.5  → 50% 旧 + 50% 新 → 平衡
#   α = 1.0  → 100% 用新值覆盖 ✓ (本代码使用简化版)
ALPHA = 1.0

## 2. Import Libraries

In [ ]:
# 中文：导入所需库
# English: Import required libraries

import os          # 用于清屏 / For screen clearing
import time        # 用于动画延迟 / For animation delay
import random      # 用于 ε-贪婪策略 / For ε-greedy policy
import importlib   # 用于动态导入模块 / For dynamic module import

print("Libraries imported successfully!")

## 3. Load Environment

The Cliff Walking environment is a 4×12 grid:
```
┌───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┐
│   │   │   │   │   │   │   │   │   │   │   │   │  Row 0
├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
│   │   │   │   │   │   │   │   │   │   │   │   │  Row 1
├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
│   │   │   │   │   │   │   │   │   │   │   │   │  Row 2
├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
│ S │ C │ C │ C │ C │ C │ C │ C │ C │ C │ C │ G │  Row 3
└───┴───┴───┴───┴───┴───┴───┴───┴───┴───┴───┴───┘
  0   1   2   3   4   5   6   7   8   9  10  11

S = Start, C = Cliff (-100 reward), G = Goal
Each step: -1 reward
```

In [ ]:
# 中文：动态导入悬崖行走环境模块
# 原因：importlib 可以导入文件名以数字开头的模块（Python 不支持 import 041107730_xxx）
# English: Dynamically import the cliff walking environment module
# Reason: importlib allows importing modules with names starting with numbers
env_module = importlib.import_module('041107730_lab2_cliff_env')

# 中文：创建悬崖行走环境（4×12 网格）
# English: Create Cliff Walking environment (4×12 grid)
env = env_module.GridEnv(size=12)

## 4. Initialize Q-table

In [ ]:
# 中文：创建 Q 表，维度为 [状态数] × [动作数]
# English: Create Q-table with dimensions [states] × [actions]
#
# 为什么用随机值初始化？
# 对比 Comparison:
#   全部初始化为 0  → 所有动作看起来一样好，可能一直选第一个
#   随机初始化 ✓    → 不同动作有不同初始值，鼓励尝试各种动作
qtable = [
    [random.random() for _ in range(env.actions())]
    for _ in range(env.states())
]

## 5. Define Core Functions

Two key functions for Q-Learning:
1. **select_action**: ε-greedy policy for exploration vs exploitation
2. **update_qtable**: Bellman equation for Q-value update

In [ ]:
def select_action(state: int, qtable: list, epsilon: float) -> int:
    """
    中文：使用 ε-贪婪策略选择动作
    English: Select action using ε-greedy policy
    
    工作原理：
      if 随机数 < ε:
          随机选动作（探索）→ 可能发现更好的路
      else:
          选 Q 值最大的动作（利用）→ 用已知最好的策略
    
    为什么需要探索？
      只利用（ε=0）→ 学到错误策略后无法纠正
      有探索（ε>0）→ 偶尔尝试其他动作，可能发现更好的路 ✓
    """
    if random.random() < epsilon:
        # 以 ε 概率随机选择（探索）
        return random.choice(range(len(qtable[state])))
    else:
        # 选择 Q 值最大的动作（利用）
        return qtable[state].index(max(qtable[state]))


def update_qtable(qtable: list, state: int, action: int, 
                  reward: float, next_state: int, gamma: float) -> None:
    """
    中文：使用贝尔曼方程更新 Q 表
    English: Update Q-table using Bellman equation
    
    公式：Q(s,a) = r + γ × max Q(s',a')
                  │   │   └── 从下一状态能获得的最大未来价值
                  │   └── 折扣因子，让未来奖励"打折"
                  └── 这一步获得的即时奖励
    
    举例：
      r = -1, max Q(s',a') = 5, γ = 0.9
      Q(s,a) = -1 + 0.9 × 5 = -1 + 4.5 = 3.5
               ↑眼前亏1   ↑未来值4.5
    """
    qtable[state][action] = reward + gamma * max(qtable[next_state])

## 6. Define Training Function

In [ ]:
def train_episode(env, qtable: list, epsilon: float, gamma: float,
                  episode_num: int, total_episodes: int) -> tuple:
    """
    中文：训练单个回合
    English: Train a single episode
    
    返回 Returns: (steps, total_reward)
    """
    # 中文：重置环境
    # English: Reset environment
    state, _, done = env.reset()
    steps = 0
    episode_reward = 0
    
    while not done:
        # 中文：清屏并显示状态
        # English: Clear screen and show state
        os.system('cls' if os.name == 'nt' else 'clear')
        print(f"Episode # {episode_num + 1} / {total_episodes}")
        print(f"Steps: {steps} | Reward: {episode_reward} | ε: {epsilon:.4f}")
        env.render()
        time.sleep(0.05)
        
        steps += 1
        
        # 中文：选择动作（ε-贪婪）
        # English: Select action (ε-greedy)
        action = select_action(state, qtable, epsilon)
        
        # 中文：执行动作，获取反馈
        # English: Execute action, get feedback
        next_state, reward, done = env.step(action)
        episode_reward += reward
        
        # 中文：更新 Q 表（贝尔曼方程）
        # English: Update Q-table (Bellman equation)
        update_qtable(qtable, state, action, reward, next_state, gamma)
        
        # 中文：转移到下一状态
        # English: Move to next state
        state = next_state
        
        # 中文：防止无限循环
        # English: Prevent infinite loops
        if steps > 1000:
            break
    
    return steps, episode_reward


print("Training function defined: train_episode()")

## 7. Execute Training

**Watch the agent learn!** The agent will explore the environment and gradually learn to avoid the cliff.

In [ ]:
# 中文：重新初始化 Q 表和探索率（确保从头开始训练）
# English: Re-initialize Q-table and epsilon (ensure training from scratch)
qtable = [
    [random.random() for _ in range(env.actions())]
    for _ in range(env.states())
]
current_epsilon = EPSILON
training_history = []

print("Starting training...\n")

# 中文：训练循环
# English: Training loop
for episode in range(EPISODES):
    # 中文：训练单个回合
    # English: Train single episode
    steps, episode_reward = train_episode(
        env, qtable, current_epsilon, GAMMA, episode, EPISODES
    )
    
    # 中文：记录历史
    # English: Record history
    training_history.append((steps, episode_reward))
    
    # 中文：衰减探索率
    # 原因：早期多探索了解环境，后期多利用已学知识
    # English: Decay exploration rate
    # Reason: More exploration early, more exploitation later
    current_epsilon -= DECAY * current_epsilon
    
    # 中文：打印回合总结
    # English: Print episode summary
    print(f"Episode {episode + 1}: {steps} steps, Reward: {episode_reward}")
    time.sleep(0.3)

print("\n" + "="*50)
print("Training complete!")
print("="*50)

## 8. Test the Trained Agent

Now we test using **pure greedy policy** (no exploration) to see the learned optimal path.

In [ ]:
def test_agent(env, qtable: list, render: bool = True) -> tuple:
    """
    中文：测试训练好的智能体（纯贪婪策略）
    English: Test trained agent (pure greedy policy)
    
    原因：测试时不需要探索，只想看智能体学到的最优策略
    Reason: No exploration needed during testing, just want to see optimal policy
    """
    state, _, done = env.reset()
    steps = 0
    total_reward = 0
    action_names = ['LEFT', 'RIGHT', 'UP', 'DOWN']
    
    if render:
        print("\n" + "="*50)
        print("Testing Trained Agent (Greedy Policy)")
        print("="*50)
        env.render()
        print()
    
    while not done and steps < 100:
        # 中文：使用纯贪婪策略（不再探索）
        # English: Use pure greedy policy (no exploration)
        action = qtable[state].index(max(qtable[state]))
        
        state, reward, done = env.step(action)
        total_reward += reward
        steps += 1
        
        if render:
            print(f"Step {steps}: {action_names[action]} | Reward: {reward}")
            env.render()
            print()
            time.sleep(0.3)
    
    return steps, total_reward, done


# 中文：测试智能体
# English: Test the agent
steps, total_reward, success = test_agent(env, qtable, render=True)

print("="*50)
print("Test Results:")
print("="*50)
print(f"  Steps taken: {steps}")
print(f"  Total Reward: {total_reward}")
print(f"  Reached Goal: {'Yes' if success else 'No'}")

## 9. Analyze Q-table

Check Q-values at key positions to verify the agent learned correctly.

In [ ]:
def analyze_qtable(qtable: list) -> None:
    """
    中文：分析关键位置的 Q 值
    English: Analyze Q-values at key positions
    
    原因：通过 Q 值可以验证智能体是否学到了合理策略
    Reason: Q-values help verify if agent learned reasonable strategy
    """
    action_names = ['LEFT', 'RIGHT', 'UP', 'DOWN']
    
    # 中文：状态编号 = 行 * 12 + 列
    # English: State number = row * 12 + col
    
    print("\n" + "="*50)
    print("Q-Table Analysis")
    print("="*50)
    
    # 中文：起始位置 (3, 0) -> state = 36
    # English: Start position (3, 0) -> state = 36
    print(f"\nStart Position (row=3, col=0), State #36:")
    print("  Expected: UP or RIGHT should have highest Q-value")
    for i, action in enumerate(action_names):
        marker = " <-- Best" if qtable[36][i] == max(qtable[36]) else ""
        print(f"  {action:6s}: {qtable[36][i]:8.2f}{marker}")
    
    # 中文：悬崖上方 (2, 1) -> state = 25
    # English: Above cliff (2, 1) -> state = 25
    print(f"\nAbove Cliff (row=2, col=1), State #25:")
    print("  Expected: DOWN should have low Q-value (leads to cliff!)")
    for i, action in enumerate(action_names):
        marker = " <-- Best" if qtable[25][i] == max(qtable[25]) else ""
        if action == 'DOWN':
            marker = " <-- Dangerous!"
        print(f"  {action:6s}: {qtable[25][i]:8.2f}{marker}")


# 中文：分析 Q 表
# English: Analyze Q-table
analyze_qtable(qtable)

## 10. Visualize Learned Policy

Display the optimal action for each state using arrows.

In [ ]:
def visualize_policy(qtable: list, rows: int = 4, cols: int = 12) -> None:
    """
    中文：用箭头可视化学到的策略
    English: Visualize learned policy with arrows
    """
    action_symbols = ['←', '→', '↑', '↓']
    
    print("\n" + "="*50)
    print("Learned Policy Visualization")
    print("="*50)
    print("\nArrow = Best action at each state")
    print("← LEFT  → RIGHT  ↑ UP  ↓ DOWN\n")
    
    for row in range(rows):
        line = ""
        for col in range(cols):
            state = row * cols + col
            
            if row == 3 and col == 0:
                line += "[S]"  # Start
            elif row == 3 and col == 11:
                line += "[G]"  # Goal
            elif row == 3 and 0 < col < 11:
                line += "[C]"  # Cliff
            else:
                best_action = qtable[state].index(max(qtable[state]))
                line += f" {action_symbols[best_action]} "
        print(line)
    
    print("\n[S]=Start  [G]=Goal  [C]=Cliff")
    print("\nExpected optimal path: Go UP first, then RIGHT along row 2, then DOWN to Goal")


# 中文：可视化学到的策略
# English: Visualize the learned policy
visualize_policy(qtable)

## 11. Training Statistics

In [ ]:
# 中文：打印训练统计
# English: Print training statistics
print("\n" + "="*50)
print("Training Statistics")
print("="*50)

steps_list = [h[0] for h in training_history]
rewards_list = [h[1] for h in training_history]

print(f"\nFirst 5 episodes (early exploration):")
for i in range(min(5, len(training_history))):
    print(f"  Episode {i+1}: {steps_list[i]:3d} steps, reward {rewards_list[i]:4d}")

print(f"\nLast 5 episodes (learned policy):")
for i in range(max(0, len(training_history)-5), len(training_history)):
    print(f"  Episode {i+1}: {steps_list[i]:3d} steps, reward {rewards_list[i]:4d}")

print(f"\nOverall Statistics:")
print(f"  Average steps: {sum(steps_list)/len(steps_list):.2f}")
print(f"  Average reward: {sum(rewards_list)/len(rewards_list):.2f}")
print(f"  Best reward: {max(rewards_list)}")
print(f"  Worst reward: {min(rewards_list)}")

print("\nConclusion:")
print("  - Early episodes: Agent falls into cliff frequently (large negative rewards)")
print("  - Later episodes: Agent learns safe path (smaller negative rewards)")
print("  - Optimal path reward: -13 (13 steps × -1 per step)")

---

## Summary

### Key Concepts Demonstrated:

1. **Q-Learning Algorithm**
   - Model-free reinforcement learning
   - Learns from trial and error
   - Uses Bellman equation: $Q(s,a) = r + \gamma \cdot \max_{a'} Q(s',a')$

2. **ε-Greedy Policy**
   - Balances exploration (trying new actions) and exploitation (using best known action)
   - ε decays over time: explore more early, exploit more later

3. **Cliff Walking Problem**
   - Classic RL benchmark
   - Agent learns to avoid cliff and find safe path to goal

### Result:
After training, the agent successfully learns to navigate above the cliff to reach the goal safely.